In [2]:
import json
import pandas as pd
import os
print(os.getcwd())
# print(os.path.abspath(base_path))


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

/Users/manilesrun/Desktop/Queryanalyser/QA/results_llm


In [3]:
# ls

In [4]:
base_path = "../results_llm"

folders = {

    3: "test_eval_service_3",
    5: "test_eval_service_5",
    8: "test_eval_service_8",
    10: "test_eval_service_10",
    12: "test_eval_service_12",
    14: "test_eval_service_14"

}

In [5]:
data = {}

for k, folder_name in folders.items():

    folder_path = os.path.join(base_path, folder_name)
#     print(folder_path)

    data[k] = []

    for file_name in os.listdir(folder_path):

        if not file_name.endswith(".json"):
            continue

        if "_efficiency" in file_name:
            continue

        file_path = os.path.join(folder_path, file_name)
#         print(file_path)

        with open(file_path, "r") as f:
            rows = json.load(f)
#             print(rows)

        for row in rows:
            row = row.copy()
            row["source_file"] = file_name
            data[k].append(row)

    print(f"K={k}: loaded {len(data[k])} rows")

K=3: loaded 1200 rows
K=5: loaded 1800 rows
K=8: loaded 2700 rows
K=10: loaded 3300 rows
K=12: loaded 3900 rows
K=14: loaded 4500 rows


In [6]:
df_k3 = pd.DataFrame(data[3])
df_k5 = pd.DataFrame(data[5])
df_k8 = pd.DataFrame(data[8])
df_k10 = pd.DataFrame(data[10])
df_k12 = pd.DataFrame(data[12])
df_k14 = pd.DataFrame(data[14])


print("K=3:", df_k3.shape)
print("K=5:", df_k5.shape)
print("K=8:", df_k8.shape)
print("K=10:", df_k10.shape)
print("K=12:", df_k12.shape)
print("K=14:", df_k14.shape)

K=3: (1200, 29)
K=5: (1800, 29)
K=8: (2700, 29)
K=10: (3300, 29)
K=12: (3900, 29)
K=14: (4500, 29)


In [7]:
# df_k3["model_name"].unique()
# df_k3.columns

## combined df

In [8]:
df_combined = pd.concat([df_k3, df_k5, df_k8, df_k10, df_k12, df_k14], ignore_index=True)

print(df_combined.shape)
print(df_combined["total_service_count"].value_counts())

(17400, 29)
14    4500
12    3900
10    3300
8     2700
5     1800
3     1200
Name: total_service_count, dtype: int64


In [9]:
df_combined.columns

Index(['routing_method', 'query', 'raw_response', 'predicted_label',
       'top_candidate', 'model_name', 'latency_ms', 'prompt_tokens',
       'completion_tokens', 'total_tokens', 'max_score', 'margin',
       'margin_ratio', 'normalized_margin', 'mean_score', 'std_score',
       'top_zscore', 'second_top_zscore', 'zscore_margin', 'scores', 'zscores',
       'confidence_threshold', 'query_id', 'expected_service', 'is_correct',
       'error', 'specialist_service_count', 'total_service_count',
       'source_file'],
      dtype='object')

In [11]:
# df = df_combined[df_combined["routing_method"] == "cross-encoder"].copy()

df = df_combined.copy()

df["overall_acc"] = (df["expected_service"] == df["predicted_label"]).astype(float)
df["specialist_recall"] = df["overall_acc"].where(df["expected_service"] != "general")
df["general_recall"]    = df["overall_acc"].where(df["expected_service"] == "general")

out = df.pivot_table(
    index=["routing_method", "model_name", "total_service_count"],
    values=["specialist_recall", "general_recall", "overall_acc"],
    aggfunc="mean",
)
out["balanced_acc"] = (out["specialist_recall"] + out["general_recall"]) / 2

out = out[["overall_acc", "specialist_recall", "general_recall", "balanced_acc"]]


pct_cols = ["specialist_recall", "general_recall", "overall_acc", "balanced_acc"]
out[pct_cols] = out[pct_cols] * 100

out.round(1)
# out.isna().sum()

overall_acc  \
routing_method model_name  total_service_count                
llm            llama3.1:8b 3                           85.0   
                           5                           81.5   
                           8                           82.1   
                           10                          78.5   
                           12                          74.2   
                           14                          71.7   
               phi4:14b    3                           56.0   
                           5                           72.2   
                           8                           82.7   
                           10                          83.0   
                           12                          79.6   
                           14                          76.7   
               qwen2.5:14b 3                           95.5   
                           5                           93.2   
                           8                           88.2   
                           10                          87.8   
                           12                          81.6   
                           14                          79.7   

                                                specialist_recall  \
routing_method model_name  total_service_count                      
llm            llama3.1:8b 3                                 95.7   
                           5                                 92.0   
                           8                                 89.2   
                           10                                83.5   
                           12                                78.1   
                           14                                74.5   
               phi4:14b    3                                 42.3   
                           5                                 66.6   
                           8                                 80.9   
                           10                                81.7   
                           12                                78.8   
                           14                                75.3   
               qwen2.5:14b 3                                 94.3   
                           5                                 93.4   
                           8                                 91.5   
                           10                                89.7   
                           12                                83.3   
                           14                                81.1   

                                                general_recall  balanced_acc  
routing_method model_name  total_service_count                                
llm            llama3.1:8b 3                              53.0          74.3  
                           5                              29.0          60.5  
                           8                              25.0          57.1  
                           10                             29.0          56.2  
                           12                             28.0          53.0  
                           14                             33.0          53.8  
               phi4:14b    3                              97.0          69.7  
                           5                             100.0          83.3  
                           8                              97.0          88.9  
                           10                             96.0          88.8  
                           12                             90.0          84.4  
                           14                             96.0          85.6  
               qwen2.5:14b 3                              99.0          96.7  
                           5                              92.0          92.7  
                           8                              62.0          76.8  
                           10                             69.0 

### inspect a model

In [69]:
MODEL = "Qwen/Qwen3-Reranker-4B"

In [70]:
d3 = df_k3[df_k3["model_name"] == MODEL].copy()

d5 = df_k5[df_k5["model_name"] == MODEL].copy()

d8 = df_k8[df_k8["model_name"] == MODEL].copy()

d10 = df_k10[df_k10["model_name"] == MODEL].copy()
d12 = df_k12[df_k12["model_name"] == MODEL].copy()
d14 = df_k14[df_k14["model_name"] == MODEL].copy()


print("K=3", d3.shape)
print("K=5", d5.shape)
print("K=8", d8.shape)

print("K=10", d10.shape)
print("K=12", d12.shape)
print("K=14", d14.shape)


K=3 (400, 42)
K=5 (600, 42)
K=8 (900, 42)
K=10 (1100, 42)
K=12 (1300, 42)
K=14 (1500, 42)


In [71]:
def expand_scores_and_zscores(df):
    """
    Expand the nested `scores` and `zscores` dict columns
    into normal dataframe columns.
    """

    scores_df = (
        pd.json_normalize(df["scores"])
        .add_suffix("_score")
    )

    zscores_df = (
        pd.json_normalize(df["zscores"])
        .add_suffix("_zscore")
    )

    df_expanded = pd.concat(
        [
            df.reset_index(drop=True),
            scores_df.reset_index(drop=True),
            zscores_df.reset_index(drop=True),
        ],
        axis=1,
    )

    return df_expanded

In [72]:
d3 = expand_scores_and_zscores(d3)
d5 = expand_scores_and_zscores(d5)
d8 = expand_scores_and_zscores(d8)
d10 = expand_scores_and_zscores(d10)
d12 = expand_scores_and_zscores(d12)
d14 = expand_scores_and_zscores(d14)

In [73]:
# d3.columns.to_list()

In [74]:
def recall_summary(df):

    specialist_df = df[
        df["expected_service"] != "general"
    ]

    general_df = df[
        df["expected_service"] == "general"
    ]

    accuracy = df["is_correct"].mean()

    specialist_recall = (
        specialist_df["predicted_label"]
        == specialist_df["expected_service"]
    ).mean()

    general_recall = (
        general_df["predicted_label"]
        == "general"
    ).mean()

    print(
        f"Accuracy: {accuracy:.4f}"
    )

    print(
        f"Specialist Recall: "
        f"{specialist_recall:.4f}"
    )

    print(
        f"General Recall: "
        f"{general_recall:.4f}"
    )


In [75]:
recall_summary(d3)
recall_summary(d5)
recall_summary(d8)
recall_summary(d10)
recall_summary(d12)
recall_summary(d14)


Accuracy: 0.9700
Specialist Recall: 0.9833
General Recall: 0.9300
Accuracy: 0.9600
Specialist Recall: 0.9640
General Recall: 0.9400
Accuracy: 0.8900
Specialist Recall: 0.8988
General Recall: 0.8200
Accuracy: 0.8809
Specialist Recall: 0.8840
General Recall: 0.8500
Accuracy: 0.8323
Specialist Recall: 0.8317
General Recall: 0.8400
Accuracy: 0.8153
Specialist Recall: 0.8150
General Recall: 0.8200


In [76]:
recall_summary(d3)

Accuracy: 0.9700
Specialist Recall: 0.9833
General Recall: 0.9300


In [77]:
cm_k3 = pd.crosstab(
    d3["expected_service"],
    d3["predicted_label"]
)

cm_k3

predicted_label,abstract_algebra,anatomy,astronomy,general
expected_service,,,,
abstract_algebra,99,0,0,1
anatomy,0,99,0,1
astronomy,0,0,97,3
general,1,4,2,93


In [78]:
error_k3 = d3[ (d3["expected_service"] == "general") & (d3["predicted_label"] == "abstract_algebra")]

In [79]:
error_k3[
    [
        "query",
        "expected_service",
        "predicted_label",

#         "abstract_algebra_zscore",
#         "anatomy_zscore",
#         "astronomy_zscore",

#         "top_zscore",
#         "second_top_zscore",
#         "zscore_margin",

#         "std_score",

        "abstract_algebra_score",
        "anatomy_score",
        "astronomy_score",
    ]
]

,query,expected_service,predicted_label,abstract_algebra_score,anatomy_score,astronomy_score
369,i want to start talking to you in swedish,general,abstract_algebra,-10.5,-11.875,-12.125


# K5

In [80]:
recall_summary(d5)

Accuracy: 0.9600
Specialist Recall: 0.9640
General Recall: 0.9400


In [81]:
cm_k5 = pd.crosstab(
    d5["expected_service"],
    d5["predicted_label"]
)
cm_k5

predicted_label,abstract_algebra,anatomy,astronomy,business_ethics,college_computer_science,general
expected_service,,,,,,
abstract_algebra,99,0,0,0,0,1
anatomy,0,99,0,0,0,1
astronomy,0,0,97,0,0,3
business_ethics,1,0,0,93,1,5
college_computer_science,5,0,0,0,94,1
general,0,2,1,1,2,94


In [82]:
# error_k5 = d5[ (d5["expected_service"] == "general") & (d5["predicted_label"] == "anatomy")]

In [83]:
# error_k5[
#     [
#         "query",
#         "expected_service",
#         "predicted_label",

#         "abstract_algebra_zscore",
#         "anatomy_zscore",
#         "astronomy_zscore",
#         "business_ethics_zscore",
#         "college_computer_science_zscore",

# #         "top_zscore",
# #         "second_top_zscore",
#         "zscore_margin",

#         "std_score",

#         "abstract_algebra_score",
#         "anatomy_score",
#         "astronomy_score",
#         "business_ethics_score",
#         "college_computer_science_score",
#     ]
# ]

# K8

In [84]:
recall_summary(d8)

Accuracy: 0.8900
Specialist Recall: 0.8988
General Recall: 0.8200


In [85]:
cm_k8 = pd.crosstab(
    d8["expected_service"],
    d8["predicted_label"]
)

cm_k8

predicted_label,abstract_algebra,anatomy,astronomy,business_ethics,college_computer_science,general,high_school_chemistry,high_school_geography,jurisprudence
expected_service,,,,,,,,,
abstract_algebra,99,0,0,0,0,1,0,0,0
anatomy,0,98,0,0,0,0,2,0,0
astronomy,0,0,97,0,0,3,0,0,0
business_ethics,0,0,0,87,1,3,0,1,8
college_computer_science,6,0,0,0,93,1,0,0,0
general,1,2,3,1,4,82,0,0,7
high_school_chemistry,2,2,1,0,0,0,95,0,0
high_school_geography,1,1,4,3,0,34,0,53,4
jurisprudence,0,0,0,0,0,3,0,0,97


In [86]:
error_k8 = d8[ (d8["expected_service"] == "general") & (d8["predicted_label"] == "college_computer_science")]

In [87]:
error_k8[
    [
        "query",
        "expected_service",
        "predicted_label",

#         "abstract_algebra_zscore",
#         "anatomy_zscore",
#         "astronomy_zscore",
#         "business_ethics_zscore",
#         "college_computer_science_zscore",
#         "high_school_chemistry_zscore",
#         "high_school_geography_zscore",
#         "jurisprudence_zscore",

#         "top_zscore",
#         "second_top_zscore",
#         "zscore_margin",

#         "std_score",
        "gap",
        "gap_cutoff",

        "abstract_algebra_score",
        "anatomy_score",
        "astronomy_score",
        "business_ethics_score",
        "college_computer_science_score",
        "high_school_chemistry_score",
        "high_school_geography_score",
        "jurisprudence_score",
    ]
]

,query,expected_service,predicted_label,gap,gap_cutoff,abstract_algebra_score,anatomy_score,astronomy_score,business_ethics_score,college_computer_science_score,high_school_chemistry_score,high_school_geography_score,jurisprudence_score
817,switch over to low power mode to preserve battery life,general,college_computer_science,2.50000,1.758946,-12.500,-11.8125,-11.6875,-13.0625,-10.1250,-12.750,-13.6875,-12.8750
864,disconnect from wifi immediately please,general,college_computer_science,2.31250,1.758946,-12.875,-11.5000,-12.2500,-13.7500,-10.6250,-13.000,-14.5000,-13.3125
888,Help! I can't find my card.,general,college_computer_science,1.78125,1.758946,-11.625,-11.1250,-11.5000,-13.1250,-10.0625,-12.375,-13.1875,-12.0625
890,i would like for my phone to be synced with you,general,college_computer_science,1.84375,1.758946,-11.000,-10.5000,-11.1875,-12.6250,-9.7500,-12.000,-13.6250,-12.0000


# K10

In [88]:
recall_summary(d10)

Accuracy: 0.8809
Specialist Recall: 0.8840
General Recall: 0.8500


In [89]:
cm_k10 = pd.crosstab(
    d10["expected_service"],
    d10["predicted_label"]
)

cm_k10

predicted_label,abstract_algebra,anatomy,astronomy,business_ethics,college_computer_science,formal_logic,general,high_school_chemistry,high_school_geography,jurisprudence,world_religions
expected_service,,,,,,,,,,,
abstract_algebra,94,0,0,0,0,6,0,0,0,0,0
anatomy,0,99,0,0,0,0,0,1,0,0,0
astronomy,0,1,97,0,0,1,1,0,0,0,0
business_ethics,0,0,0,84,1,1,2,0,0,8,4
college_computer_science,5,0,0,0,90,4,1,0,0,0,0
formal_logic,0,0,0,0,0,96,1,0,0,1,2
general,0,1,3,1,3,0,85,0,0,6,1
high_school_chemistry,0,2,0,0,0,0,0,98,0,0,0
high_school_geography,1,0,2,4,0,1,26,1,47,5,13


In [90]:
error_k10 = d10[ (d10["expected_service"] == "high_school_geography") & (d10["predicted_label"] == "general")]

In [91]:
error_k10[
    [
        "query",
        "expected_service",
        "predicted_label",

#         "abstract_algebra_zscore",
#         "anatomy_zscore",
#         "astronomy_zscore",
#         "business_ethics_zscore",
#         "college_computer_science_zscore",
#         "formal_logic_zscore",
#         "high_school_chemistry_zscore",
#         "high_school_geography_zscore",
#         "jurisprudence_zscore",
#         "world_religions_zscore",

#         "top_zscore",
#         "second_top_zscore",
#         "zscore_margin",

#         "std_score",

        "abstract_algebra_score",
        "anatomy_score",
        "astronomy_score",
        "business_ethics_score",
        "college_computer_science_score",
        "formal_logic_score",
        "high_school_chemistry_score",
        "high_school_geography_score",
        "jurisprudence_score",
        "world_religions_score",
    ]
]

,query,expected_service,predicted_label,abstract_algebra_score,anatomy_score,astronomy_score,business_ethics_score,college_computer_science_score,formal_logic_score,high_school_chemistry_score,high_school_geography_score,jurisprudence_score,world_religions_score
701,"Olives, grapes, and chickpeas are staple crops of which type of farming?",high_school_geography,general,-13.6875,-12.1250,-12.6875,-13.7500,-13.3750,-14.3125,-12.7500,-11.6250,-13.5625,-12.8125
702,The majority of Kurds are found in which country?,high_school_geography,general,-14.0625,-13.1875,-13.6250,-14.1875,-13.8750,-14.3125,-13.7500,-12.1250,-14.0000,-12.6875
707,A map that presents a single class of statistics is called a,high_school_geography,general,-10.2500,-11.2500,-10.0625,-12.0625,-10.6250,-11.0000,-11.8750,-10.3750,-11.5000,-12.3125
715,A chief factor that contributed to the Second Agricultural Revolution was,high_school_geography,general,-11.6250,-12.6875,-11.8750,-13.1250,-12.8125,-13.4375,-12.9375,-12.6875,-12.2500,-11.9375
720,Which of the following statements is NOT part of Gunnar Myrdal's theory of cumulative causation?,high_school_geography,general,-11.6875,-12.9375,-11.8125,-12.5625,-11.3750,-11.5000,-12.6250,-11.6875,-10.0000,-11.9375
721,"Part of the boundary between the United States and Mexico is the Rio Grande, an example of",high_school_geography,general,-12.3125,-11.0000,-10.4375,-13.4375,-12.7500,-13.3750,-12.0625,-11.0000,-10.2500,-11.3125
726,The purpose for the creation of NATO was to,high_school_geography,general,-12.6875,-12.7500,-12.2500,-13.3750,-12.6250,-13.2500,-13.4375,-13.3750,-12.5000,-12.1875
729,Which factor in the last quarter century greatly destabilized Eastern Europe and led to ethnic conflict?,high_school_geography,general,-12.0000,-12.1875,-12.3750,-12.4375,-13.0625,-14.0000,-12.9375,-12.0625,-11.8750,-11.5625
730,Opponents of genetically modified organisms (GMOs) in food are afraid that the GMOs,high_school_geography,general,-13.0625,-11.2500,-11.8750,-12.0000,-12.8750,-12.5000,-12.0625,-13.3750,-11.4375,-12.1250
731,What family do most European languages belong to?,high_school_geography,general,-13.0000,-12.6250,-12.8750,-14.1250,-13.1875,-12.5625,-13.3125,-11.6875,-13.1250,-11.9375


In [92]:
cm_k12 = pd.crosstab(
    d12["expected_service"],
    d12["predicted_label"]
)

cm_k12

predicted_label,abstract_algebra,anatomy,astronomy,business_ethics,college_computer_science,formal_logic,general,high_school_chemistry,high_school_geography,jurisprudence,marketing,sociology,world_religions
expected_service,,,,,,,,,,,,,
abstract_algebra,94,0,0,0,0,6,0,0,0,0,0,0,0
anatomy,0,98,0,0,0,0,0,2,0,0,0,0,0
astronomy,1,1,97,0,0,1,0,0,0,0,0,0,0
business_ethics,0,0,0,77,1,1,2,0,0,7,7,1,4
college_computer_science,5,0,0,0,90,4,1,0,0,0,0,0,0
formal_logic,0,0,0,0,0,97,0,0,0,1,0,0,2
general,0,2,3,1,3,0,84,0,0,6,0,0,1
high_school_chemistry,0,2,0,0,0,0,0,98,0,0,0,0,0
high_school_geography,1,0,2,2,0,1,24,1,43,6,1,7,12


In [98]:
error_k12 = d12[ (d12["expected_service"] == "general") & (d12["predicted_label"] == "jurisprudence")]

In [99]:
error_k12.columns.to_list()

['routing_method',
 'query',
 'a',
 'b',
 'raw_response',
 'predicted_label',
 'top_candidate',
 'model_name',
 'latency_ms',
 'prompt_tokens',
 'completion_tokens',
 'total_tokens',
 'max_score',
 'margin',
 'mean_score',
 'std_score',
 'top_zscore',
 'second_top_zscore',
 'zscore_margin',
 'scores',
 'zscores',
 'z_max',
 'winner_strength_percentage',
 'required_strength_percentage',
 'automatic_threshold',
 'strength_passed',
 'apply_threshold',
 'gate_mode',
 'mad_trust',
 'mad_cutoff',
 'mad_tau',
 'mad_passed',
 'gap',
 'gap_cutoff',
 'gap_passed',
 'query_id',
 'expected_service',
 'is_correct',
 'error',
 'specialist_service_count',
 'total_service_count',
 'source_file',
 'abstract_algebra_score',
 'anatomy_score',
 'astronomy_score',
 'business_ethics_score',
 'college_computer_science_score',
 'high_school_chemistry_score',
 'high_school_geography_score',
 'jurisprudence_score',
 'world_religions_score',
 'formal_logic_score',
 'marketing_score',
 'sociology_score',
 'abstra

In [103]:
error_k12[
    [
        "query",
        "expected_service",
        "predicted_label",
        
              "gap", 
        "gap_cutoff",
        
'abstract_algebra_score',
 'anatomy_score',
 'astronomy_score',
 'business_ethics_score',
 'college_computer_science_score',
 'high_school_chemistry_score',
 'high_school_geography_score',
 'jurisprudence_score',
 'world_religions_score',
 'formal_logic_score',
 'marketing_score',
 'sociology_score'
    ]
]

,query,expected_service,predicted_label,gap,gap_cutoff,abstract_algebra_score,anatomy_score,astronomy_score,business_ethics_score,college_computer_science_score,high_school_chemistry_score,high_school_geography_score,jurisprudence_score,world_religions_score,formal_logic_score,marketing_score,sociology_score
1204,Can I change the amount I made on a payment that I made to the payment is correct.,general,jurisprudence,2.50000,1.919956,-10.3750,-10.7500,-11.5000,-10.1250,-11.1250,-12.0000,-13.1875,-8.5000,-11.2500,-9.7500,-11.9375,-10.8750
1207,my capital one visa was taken without permission,general,jurisprudence,2.65625,1.919956,-12.3750,-11.7500,-12.5000,-12.6250,-12.0625,-13.5000,-13.8750,-10.0625,-12.8125,-12.8125,-13.6250,-12.9375
1245,I would like to know why I was charged twice for my purchase.,general,jurisprudence,2.00000,1.919956,-12.7500,-11.7500,-12.1250,-11.5625,-11.8750,-12.6875,-13.3750,-10.0000,-12.9375,-11.6250,-11.1875,-12.3750
1256,I took out a foreign currency and the exchange rate is wrong.,general,jurisprudence,2.71875,1.919956,-10.7500,-11.0000,-11.4375,-11.2500,-10.8125,-10.8750,-12.2500,-8.5000,-12.1250,-11.1875,-13.0000,-11.8750
1258,I understand that this is extremely late. After going over my transactions for the last couple of months I noticed a large charge that I'm sure I did not make. Is there anyway I can still dispute this?,general,jurisprudence,2.59375,1.919956,-13.0625,-12.0625,-12.7500,-12.5000,-12.6875,-12.6250,-13.6875,-10.1250,-12.7500,-12.6875,-13.8750,-12.8750
1277,I wasn't applied the correct exchange rate for an item I bought,general,jurisprudence,3.09375,1.919956,-9.6875,-11.1250,-10.3125,-10.0000,-10.5000,-9.5625,-10.6875,-7.5000,-11.2500,-11.1250,-11.1250,-10.8125
